# Beca 18 RAG Chatbot
## Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC

In [1]:
# STEP 0: Setup
import os
import sys
import time
import random
import tiktoken
import chromadb
import ipywidgets as widgets
from IPython.display import display, clear_output
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai
from google.genai import types
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found in .env"

client = genai.Client(api_key=GEMINI_API_KEY)

print("pypdf:", __import__('pypdf').__version__)
print("tiktoken:", tiktoken.__version__)
print("chromadb:", chromadb.__version__)
print("ipywidgets:", widgets.__version__)
print("\u2713 GEMINI_API_KEY loaded")

pypdf: 6.11.0
tiktoken: 0.12.0
chromadb: 1.5.9
ipywidgets: 8.1.8
✓ GEMINI_API_KEY loaded


In [2]:
# STEP 1: PDF Text Extraction
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = " ".join(text.split())
        full_text += f"\n[PAGE {i+1}]\n{text}\n"
    return full_text

pdf_path = "../data/beca18_reglamento.pdf"
full_text = extract_text_from_pdf(pdf_path)

print(f"Total characters: {len(full_text):,}")
print(f"Total words: {len(full_text.split()):,}")
print(f"\nFirst 500 chars:\n{full_text[:500]}")

Total characters: 366,366
Total words: 55,202

First 500 chars:

[PAGE 1]
Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS: El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ de la Oficina de Asesoría Jurídica, y; CONSIDERANDO: Que, la Ley N° 29837 crea el Pro


In [ ]:
# STEP 2: Tokenization and Chunking
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(full_text)
print(f"Total tokens: {len(tokens):,}")
print(f"Embedding limit: 8,192 tokens")
print(f"Chunk size 800 tokens with 100 overlap is appropriate because:")
print(f"  - Each chunk fits well within the 8,192 token embedding limit")
print(f"  - Overlap preserves context between adjacent chunks")
print(f"  - Small enough to be specific, large enough to be meaningful")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_text(full_text)
metadata = [{"document": "beca18_reglamento", "topic": "Beca18", "language": "es"} 
            for _ in chunks]

print(f"\nTotal chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks)/len(chunks):.0f} chars")

In [4]:
# STEP 3: Embeddings
def embed_documents(texts, max_retries=5):
    embeddings = []
    for i, text in enumerate(tqdm(texts, desc="Embedding docs")):
        for attempt in range(max_retries):
            try:
                result = client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=text,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                )
                embeddings.append(result.embeddings[0].values)
                time.sleep(1.1)
                break
            except Exception as e:
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"Retry {attempt+1} for chunk {i}: {e}. Waiting {wait:.1f}s")
                time.sleep(wait)
    return embeddings

def embed_query(text, max_retries=5):
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model="gemini-embedding-001",
                contents=text,
                config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
            )
            return result.embeddings[0].values
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"Retry {attempt+1}: {e}. Waiting {wait:.1f}s")
            time.sleep(wait)
    raise Exception("Max retries exceeded")

print("\u2713 Embedding functions defined with exponential backoff")

✓ Embedding functions defined with exponential backoff


In [5]:
# STEP 4: Vector Database
chroma_client = chromadb.PersistentClient(path="chroma_db_beca18")

collection_name = "beca18_chunks"
existing = [c.name for c in chroma_client.list_collections()]

if collection_name in existing:
    collection = chroma_client.get_collection(collection_name)
    print(f"\u2713 Loaded existing collection with {collection.count()} documents")
else:
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )
    print("Indexing chunks (this may take a few minutes)...")
    embeddings = embed_documents(chunks)
    ids = [f"chunk_{i}" for i in range(len(chunks))]
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadata
    )
    print(f"\u2713 Indexed {collection.count()} documents")

Indexing chunks (this may take a few minutes)...


Embedding docs:  35%|██████████████████████████████████████████▋                                                                              | 512/1451 [12:50<24:02,  1.54s/it]

Retry 1 for chunk 512: 502 Bad Gateway. {'message': '<!DOCTYPE html>\n<html lang=en>\n  <meta charset=utf-8>\n  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">\n  <title>Error 502 (Server Error)!!1</title>\n  <style>\n    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/branding/googlelogo/1x/googlelogo_color_150x54dp.png) no-repeat;margin-left:-5px}@media only screen and (min-resolution:192dpi){#logo{background:url(//www.google.com/images/branding/googlelogo/2x/googlelogo_color_150x54

Embedding docs:  67%|████████████████████████████████████████████████████████████████████████████████▉                                        | 970/1451 [24:43<12:27,  1.55s/it]

Retry 1 for chunk 970: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 55.216866097s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model':

...

Retry 1 for chunk 1006: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 54.103918188s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model'

Embedding docs:  69%|████████████████████████████████████████████████████████████████████████████████▌                                   | 1007/1451 [1:01:19<3:42:06, 30.01s/it]

Retry 1 for chunk 1007: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 19.266816081s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model'

Embedding docs:  69%|█████████████████████████████████████████████████████████████████████████████████▉                                    | 1007/1451 [1:01:28<27:06,  3.66s/it]


KeyboardInterrupt: 

In [ ]:
# STEP 5: Semantic Search
def semantic_search(question, k=5):
    query_embedding = embed_query(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    output = []
    for i in range(len(results["documents"][0])):
        output.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })
    return output

test_question = "\u00bfCu\u00e1les son los requisitos para postular a Beca 18?"
results = semantic_search(test_question, k=3)
print(f"Top 3 results for: '{test_question}'\n")
for i, r in enumerate(results):
    print(f"[{i+1}] Distance: {r['distance']:.4f}")
    print(f"     Text: {r['text'][:200]}...")
    print()

In [ ]:
# STEP 6: Grounded Generation
SYSTEM_PROMPT = """Eres un asistente experto en el reglamento de Beca 18 (PRONABEC).
Responde \u00daNICAMENTE bas\u00e1ndote en el contexto proporcionado.
Cuando cites informaci\u00f3n, menciona el n\u00famero de p\u00e1gina si est\u00e1 disponible usando [PAGE N].
Si la informaci\u00f3n no est\u00e1 en el contexto, responde exactamente: 
\"El documento no contiene informaci\u00f3n sobre este tema.\"
No uses conocimiento previo ni hagas suposiciones."""

def answer_with_context(question, k=5):
    results = semantic_search(question, k=k)
    context = "\n\n".join([
        f"[Fragmento {i+1} | Distancia: {r['distance']:.3f}]\n{r['text']}"
        for i, r in enumerate(results)
    ])
    prompt = f"""Contexto del reglamento Beca 18:
{context}

Pregunta: {question}

Responde bas\u00e1ndote exclusivamente en el contexto anterior."""
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT),
        contents=prompt
    )
    return response.text, results

test_questions = [
    "\u00bfCu\u00e1les son los requisitos de elegibilidad para postular a Beca 18?",
    "\u00bfCu\u00e1les son las modalidades de la beca?",
    "\u00bfCu\u00e1l es el monto del est\u00edpendio mensual?",
    "\u00bfCu\u00e1les son las obligaciones del becario?",
    "\u00bfEn qu\u00e9 casos se puede perder la beca?",
    "\u00bfCu\u00e1l es el precio del iPhone 16 Pro Max?"
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {q}")
    print(f"{'='*60}")
    answer, _ = answer_with_context(q)
    print(f"RESPUESTA: {answer}")
    time.sleep(2)

In [ ]:
# STEP 7: Interactive Chat Interface
output_area = widgets.Output()
question_input = widgets.Text(
    placeholder="Escribe tu pregunta sobre Beca 18...",
    layout=widgets.Layout(width="100%")
)
ask_button = widgets.Button(
    description="Consultar",
    button_style="primary",
    layout=widgets.Layout(width="150px")
)
clear_button = widgets.Button(
    description="Limpiar",
    button_style="warning",
    layout=widgets.Layout(width="150px")
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="Fragmentos:",
    style={"description_width": "initial"}
)

def on_ask(b):
    question = question_input.value.strip()
    if not question:
        return
    with output_area:
        clear_output()
        print(f"\U0001f50d Buscando respuesta para: {question}\n")
        try:
            answer, sources = answer_with_context(question, k=k_slider.value)
            print(f"\U0001f4ac Respuesta:\n{answer}\n")
            accordion_children = []
            for i, s in enumerate(sources):
                text_widget = widgets.HTML(
                    f"<b>P\u00e1gina:</b> (ver texto) | <b>Distancia:</b> {s['distance']:.4f}<br><br>{s['text']}"
                )
                accordion_children.append(text_widget)
            accordion = widgets.Accordion(children=accordion_children)
            for i in range(len(sources)):
                accordion.set_title(i, f"Fuente {i+1} \u2014 distancia: {sources[i]['distance']:.4f}")
            accordion.selected_index = None
            print("\n\U0001f4da Fuentes utilizadas:")
            display(accordion)
        except Exception as e:
            print(f"\u274c Error: {e}")

def on_clear(b):
    question_input.value = ""
    with output_area:
        clear_output()

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

ui = widgets.VBox([
    widgets.HTML("<h2>\U0001f393 Chatbot Normativa Beca 18</h2><p>Resoluci\u00f3n Directoral Ejecutiva N.\u00b0 033-2026-MINEDU/VMGI-PRONABEC</p>"),
    question_input,
    widgets.HBox([ask_button, clear_button, k_slider]),
    output_area
])
display(ui)